# ConformalStereo: fetch and parse the Cambridge deposits

This notebook downloads public deposits of the Goodman group from Apollo, the University of Cambridge repository,
reads every Gaussian output and Maestro file **inside the archives**, and writes small result files.
Nothing has to be unpacked by hand, and the archives are deleted from the scratch disk as soon as they are read.

**Before running**
1. Right panel, *Settings*: switch **Internet** on. Accelerator: None.
2. *Save Version*, then **Save & Run All (Commit)**. The run needs no attention.
   Expect under an hour; most of it is download time and depends on the repository.

**After the run**
Open the saved version, go to *Output*, download every file named `cs_core_NN.zip` and send them back.
The `cs_extra_NN.zip` files hold conformer geometries; keep them for later.

| Key | Deposit | DOI | Default |
| --- | --- | --- | --- |
| `dp4ai` | DP4-AI: 46 stereo cases at three levels of theory | 10.17863/CAM.47721 | download and parse |
| `dp5` | DP5, revised: structure reassignment examples | 10.17863/CAM.81332 | download and parse |
| `conf2018` | Conformational search and DFT data (Ermanis et al., 2018) | 10.17863/CAM.32463 | download and parse |
| `dp4x2` | "Doubling the power of DP4" (2017): 42 compounds | 10.17863/CAM.13222 | list the files only |

The record pages give CC BY 4.0 as the licence of these deposits; cite the DOIs when the data are used.

What is written for every Gaussian job step: route, program version, charge, multiplicity, last SCF energy,
free energy if printed, termination and optimisation flags, CPU time, and the isotropic shielding of every atom.
For every Maestro structure: the MacroModel energies. Structures, NMR description files and other small text files
are copied as they are. A full inventory of every archive is included, so that nothing depends on a guess about
the folder layout.

Only the cell "1. Settings" is meant to be edited.

In [ ]:
# ---- 1. Settings ---------------------------------------------------------------------------------------------
import os, sys, re, io, json, gzip, time, math, shutil, hashlib, zipfile, tarfile, traceback, collections
import subprocess, multiprocessing

NOTEBOOK_VERSION = '1.0 (2026-11-13)'

# mode: 'parse' = download and parse | 'list' = write the file listing only | 'off' = ignore
DEPOSITS = [
    dict(key='dp4ai', mode='parse', doi='10.17863/CAM.47721', handle='1810/303161', item='3b9dadf2-4701-49af-af2f-c8491384e199',
         title='DP4-AI: DFT, molecular mechanics and raw NMR data (46 stereo cases, three levels)',
         skip_names=['NMR_Data.zip']),                      # raw Bruker spectra, 278 MB, not needed here
    dict(key='dp5', mode='parse', doi='10.17863/CAM.81332', handle='1810/334048', item='f4786c07-e546-4e0a-b8fb-c8022997c9de',
         title='DP5, revised: structure reassignment examples'),
    dict(key='conf2018', mode='parse', doi='10.17863/CAM.32463', handle='1810/285709', item='53d5cb49-ed72-439d-99b4-7c6e3586e5cb',
         title='Ermanis 2018: conformational search and DFT data at several geometry and energy levels'),
    dict(key='dp4x2', mode='list', doi='10.17863/CAM.13222', handle='1810/267217', item='b41c395b-8a42-43be-b2f9-ee39522f1133',
         title='Doubling the power of DP4 (2017): 42 compounds, DFT conditions screened'),
]

ONLY_FILES = []            # e.g. ['AT1.zip', 'AT2.zip'] for a quick trial; empty = every file of the deposit
LOCAL_SOURCES = {}         # manual route: {'dp4ai': ['/kaggle/input/my-upload']} = folders or archives already on disk
MAX_FILE_GB = 12           # a single repository file above this size is skipped and reported
MAX_DEPOSIT_GB = 40        # a deposit above this total size is listed, not downloaded
DEADLINE_HOURS = 10.5      # stop starting new archives after this long and package what is done
PART_MB = 20               # size of each result zip
MAE_WINDOW_KJ = 12.0       # conformer geometries are kept for Maestro structures within this energy of the minimum
MAE_MAX_GEOMS = 500        # ... and at most this many per Maestro file
SMALL_FILE_KB = 300        # text files up to this size (structures, NMR descriptions, DP4 outputs) are copied
SMALL_TOTAL_MB = 80        # ... up to this total per deposit, after removing duplicates
BIG_MEMBER_MB = 300        # outputs larger than this are read as first 2 MB + last 64 MB only

# Test hooks: environment variables override the defaults above. Not needed on Kaggle.
ON_KAGGLE = os.path.isdir('/kaggle/working')
WORK = os.environ.get('CS_WORK', '/kaggle/working' if ON_KAGGLE else os.path.abspath('cs_work'))
API_BASES = [b.strip().rstrip('/') for b in os.environ.get(
    'CS_API_BASES', 'https://api.repository.cam.ac.uk/server/api,https://www.repository.cam.ac.uk/server/api').split(',') if b.strip()]
UI_BASE = os.environ.get('CS_UI_BASE', 'https://www.repository.cam.ac.uk').rstrip('/')
INPUT_DIRS = [d for d in os.environ.get('CS_INPUT_DIRS', '/kaggle/input').split(os.pathsep) if d]
PART_MB = float(os.environ.get('CS_PART_MB', PART_MB))
if os.environ.get('CS_DEPOSITS_JSON'):
    DEPOSITS = json.loads(os.environ['CS_DEPOSITS_JSON'])
if os.environ.get('CS_LOCAL_SOURCES_JSON'):
    LOCAL_SOURCES = json.loads(os.environ['CS_LOCAL_SOURCES_JSON'])
N_WORKERS = int(os.environ.get('CS_WORKERS', max(1, min(4, os.cpu_count() or 1))))
if os.environ.get('CS_OVERRIDES_JSON'):
    globals().update(json.loads(os.environ['CS_OVERRIDES_JSON']))

# Files known from the record pages on 13 Nov 2026: used for a size check, and as a fallback list if the API is unreachable.
KNOWN_FILES = json.loads(os.environ['CS_KNOWN_FILES_JSON']) if os.environ.get('CS_KNOWN_FILES_JSON') else {
    'dp4ai': [('AT1.zip', 'aa3338a6-bb07-4a8e-814f-899c56093f6c', 94770185),
              ('AT2.zip', '6a6ffaaa-5343-49bf-97c2-6aafc31a7c71', 21216229),
              ('AT3.zip', '07c9da8c-a09c-4959-b984-94422c5565cf', 30462336),
              ('BYH1.zip', '5f545c9e-e1f4-4af9-acd8-3f570e8754f4', 59824360),
              ('BYH2.zip', 'be1587ad-d320-4e0f-946f-2d94e3a6200f', 90361606)],
    'dp5': [('Structure_Reassignment_Examples_Final.zip', '0bca6327-c8ac-40da-9305-48d968655406', 352224171)],
}

In [ ]:
# ---- 2. Folders, log, small helpers ---------------------------------------------------------------------------
T0 = time.time()


def pick_scratch():
    """Scratch folder with the most free space. Archives live here and are deleted after parsing."""
    if os.environ.get('CS_SCRATCH'):
        return os.environ['CS_SCRATCH']
    best, free = None, -1
    for cand in (('/kaggle/temp', '/tmp', WORK) if ON_KAGGLE else ('/tmp', WORK)):
        try:
            os.makedirs(cand, exist_ok=True)
            f = shutil.disk_usage(cand).free
            if f > free:
                best, free = cand, f
        except OSError:
            pass
    return os.path.join(best, 'cs_scratch')


SCRATCH = pick_scratch()
DL = os.path.join(SCRATCH, 'downloads')       # repository files
TMP = os.path.join(SCRATCH, 'tmp')            # nested archives, extracted trees
OUT = os.path.join(SCRATCH, 'out')            # result files before packaging
CORE, EXTRA = os.path.join(OUT, 'core'), os.path.join(OUT, 'extra')
for d in (WORK, DL, TMP, CORE, EXTRA):
    os.makedirs(d, exist_ok=True)
LOG_PATH = os.path.join(CORE, 'run_log.txt')


def log(*parts):
    line = '[%6.0fs] %s' % (time.time() - T0, ' '.join(str(p) for p in parts))
    print(line, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as fh:
        fh.write(line + '\n')


def human(n):
    n = float(n or 0)
    for unit in ('B', 'kB', 'MB', 'GB', 'TB'):
        if abs(n) < 1024 or unit == 'TB':
            return ('%.0f %s' if unit == 'B' else '%.1f %s') % (n, unit)
        n /= 1024


def free_gb(path):
    return shutil.disk_usage(path).free / 1024 ** 3


def hours():
    return (time.time() - T0) / 3600


def safe_name(s):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', s)[:120] or 'x'


log('notebook version', NOTEBOOK_VERSION, '| python', sys.version.split()[0], '| workers', N_WORKERS)
log('results ->', WORK, '| scratch ->', SCRATCH, '| free there: %.1f GB' % free_gb(SCRATCH))

In [ ]:
# ---- 3. Repository: list the files of a record and download them ------------------------------------------------
import requests

SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'ConformalStereo-notebook/1.0 (reuse of open research data; python-requests)',
                        'Accept': 'application/json, */*'})
SKIP_BUNDLES = {'LICENSE', 'THUMBNAIL', 'TEXT', 'CC-LICENSE', 'BRANDED_PREVIEW', 'SWORD', 'METADATA', 'ORE'}
DEAD_BASES = set()                                            # API addresses that did not answer at all
DEAD_HOSTS = set()                                            # download hosts that could not be connected to


def _retry_wait(resp, attempt):
    try:
        ra = float(resp.headers.get('Retry-After', '')) if resp is not None else 0.0
    except ValueError:
        ra = 0.0
    return min(ra or 2.0 ** (attempt + 1), 120.0)


class Unreachable(RuntimeError):
    pass


def http_json(url, params=None, tries=4):
    last = 'no attempt'
    for attempt in range(tries):
        try:
            r = SESSION.get(url, params=params, timeout=(20, 90))
        except requests.RequestException as exc:
            last = repr(exc)[:200]
            if attempt + 1 < tries:
                time.sleep(min(2.0 ** (attempt + 1), 30))
            continue
        if r.status_code in (429, 500, 502, 503, 504):
            last = 'HTTP %d' % r.status_code
            time.sleep(_retry_wait(r, attempt))
            continue
        if r.status_code != 200:
            raise RuntimeError('HTTP %d for %s: %r' % (r.status_code, r.url, r.text[:160]))
        try:
            return r.json()
        except ValueError:
            raise RuntimeError('not JSON from %s (content-type %s): %r' % (r.url, r.headers.get('Content-Type'), r.text[:160]))
    raise Unreachable('no answer from %s after %d tries: %s' % (url, tries, last))


def paged(url, key, page_size=100, max_pages=400):
    """All embedded objects of a paginated DSpace list (follows the 'next' link, or counts pages)."""
    items, seen, params, page_no = [], set(), {'size': page_size}, 0
    for _ in range(max_pages):
        mark = (url, json.dumps(params, sort_keys=True))
        if mark in seen:
            break
        seen.add(mark)
        j = http_json(url, params=params)
        items.extend((j.get('_embedded') or {}).get(key) or [])
        page = j.get('page') or {}
        number = page.get('number', page_no)
        nxt = ((j.get('_links') or {}).get('next') or {}).get('href')
        if nxt:
            url, params, page_no = nxt, None, number + 1      # 'next' links already carry the query
        elif page.get('totalPages') is not None and number + 1 < page['totalPages']:
            page_no = number + 1
            url, params = url.split('?')[0], {'size': page.get('size') or page_size, 'page': page_no}
        else:
            break
    return items


def bundle_addresses(base, dep):
    """Where the bundle list of a record may be: by item id first, then through its handle."""
    if dep.get('item'):
        yield '%s/core/items/%s/bundles' % (base, dep['item'])
    for pid in (('hdl:' + dep['handle'], dep['handle']) if dep.get('handle') else ()):
        try:
            r = SESSION.get('%s/pid/find' % base, params={'id': pid}, timeout=(20, 60))     # answers with a redirect to the item
            j = r.json() if r.status_code == 200 else {}
        except (requests.RequestException, ValueError):
            continue
        href = ((j.get('_links') or {}).get('bundles') or {}).get('href')
        uuid = j.get('uuid') or j.get('id')
        if href or uuid:
            yield href or '%s/core/items/%s/bundles' % (base, uuid)
            return


def list_record(dep):
    """Files of one record: [{name, size, md5, uuid, urls}], and how the list was obtained."""
    errors = []
    for base in API_BASES:
        if base in DEAD_BASES:
            errors.append('%s: skipped, it did not answer earlier in this run' % base)
            continue
        for address in bundle_addresses(base, dep):
            try:
                bundles = paged(address, 'bundles')
                wanted = [b for b in bundles if b.get('name') == 'ORIGINAL'] or \
                         [b for b in bundles if b.get('name') not in SKIP_BUNDLES]
                files = []
                for b in wanted:
                    href = ((b.get('_links') or {}).get('bitstreams') or {}).get('href') or \
                           '%s/core/bundles/%s/bitstreams' % (base, b.get('uuid') or b.get('id'))
                    for bs in paged(href, 'bitstreams'):
                        uuid = bs.get('uuid') or bs.get('id')
                        cs = bs.get('checkSum') or {}
                        content = ((bs.get('_links') or {}).get('content') or {}).get('href')
                        urls = [u for u in (content, '%s/core/bitstreams/%s/content' % (base, uuid),
                                            '%s/bitstreams/%s/download' % (UI_BASE, uuid)) if u]
                        files.append(dict(name=bs.get('name') or uuid, size=bs.get('sizeBytes'), uuid=uuid, bundle=b.get('name'),
                                          md5=cs.get('value') if str(cs.get('checkSumAlgorithm', '')).upper() == 'MD5' else None,
                                          urls=list(dict.fromkeys(urls))))
                if files:
                    return files, 'api:' + base
                errors.append('%s: record found but no files in bundles %s' % (address, [b.get('name') for b in bundles]))
            except Unreachable as exc:
                DEAD_BASES.add(base)
                errors.append('%s: %s' % (base, str(exc)[:300]))
                break
            except Exception as exc:                          # try the next address, then the fallback list
                errors.append('%s: %s' % (address, str(exc)[:300]))
    for e in errors:
        log('   listing failed |', e)
    known = KNOWN_FILES.get(dep['key']) or []
    files = [dict(name=n, size=sz, uuid=u, bundle='ORIGINAL', md5=None,
                  urls=['%s/core/bitstreams/%s/content' % (b, u) for b in API_BASES] + ['%s/bitstreams/%s/download' % (UI_BASE, u)])
             for n, u, sz in known]
    return files, 'fallback list (incomplete)' if files else 'none'


def md5_of(path, block=1 << 22):
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(block), b''):
            h.update(chunk)
    return h.hexdigest()


def _fetch(url, part, pos):
    """One request. Returns (state, note, total size if the server told it, seconds to wait).
    state: 'done' = stream ended cleanly | 'retry' = transient | 'restart' = drop the partial file | 'fatal' = this address is no good."""
    headers = {'Accept': '*/*'}
    if pos:
        headers['Range'] = 'bytes=%d-' % pos
    try:
        with SESSION.get(url, headers=headers, stream=True, timeout=(20, 180)) as r:
            code = r.status_code
            if code in (429, 500, 502, 503, 504):
                return 'retry', 'HTTP %d' % code, None, _retry_wait(r, 0)
            if code == 416:
                return 'restart', 'range refused', None, 0
            if code not in (200, 206):
                return 'fatal', 'HTTP %d from %s' % (code, url), None, 0
            if 'text/html' in (r.headers.get('Content-Type') or '').lower():
                return 'fatal', 'a web page came back instead of the file from %s' % url, None, 0
            total = None
            m = re.match(r'bytes\s+\d+-\d+/(\d+)', r.headers.get('Content-Range') or '')
            if code == 206 and m:
                total = int(m.group(1))
            elif code == 200 and (r.headers.get('Content-Length') or '').isdigit() and not r.headers.get('Content-Encoding'):
                total = int(r.headers['Content-Length'])
            with open(part, 'ab' if (pos and code == 206) else 'wb') as fh:
                for chunk in r.iter_content(1 << 16):         # small chunks: a broken connection loses little
                    if chunk:
                        fh.write(chunk)
            return 'done', '', total, 0
    except (requests.RequestException, OSError) as exc:
        return 'retry', repr(exc)[:160], None, -1             # -1: the connection itself failed


def download(f, dest, tries=5):
    """Download one repository file with resume and checks. Returns (ok, note)."""
    part = dest + '.part'
    note = 'no reachable address: the repository did not answer earlier in this run'
    for url in f['urls']:
        host = '/'.join(url.split('/')[:3])
        if host in DEAD_HOSTS or any(url.startswith(b) for b in DEAD_BASES):
            continue
        if os.path.exists(part):
            os.remove(part)
        expected, stalls, md5_retry, answered = f.get('size'), 0, False, False
        for _ in range(300):
            if stalls >= tries:
                if not answered:
                    DEAD_HOSTS.add(host)                      # never connected: do not wait for this host again
                break
            pos = os.path.getsize(part) if os.path.exists(part) else 0
            state, msg, total, wait = _fetch(url, part, pos)
            answered = answered or wait != -1
            wait = max(wait, 0)
            note = msg or note
            if total and not expected:
                expected = total
            got = os.path.getsize(part) if os.path.exists(part) else 0
            stalls = 0 if got > pos else stalls + 1
            if state == 'fatal':
                break
            if state == 'restart':
                if os.path.exists(part):
                    os.remove(part)
                continue
            if state == 'retry':
                time.sleep(wait or min(2.0 ** stalls, 30))
                continue
            if expected and got < expected:
                note = 'connection closed at %s of %s' % (human(got), human(expected))
                continue                                      # resume from where it stopped
            if expected and got > expected:
                note = 'more bytes than expected'
                os.remove(part)
                continue
            if f.get('md5'):
                if md5_of(part) != f['md5'].lower():
                    note = 'checksum mismatch'
                    os.remove(part)
                    if md5_retry:
                        break
                    md5_retry = True
                    continue
                check = 'md5 ok'
            else:
                check = 'size ok' if expected else 'no size or checksum to compare'
            os.replace(part, dest)
            return True, check
    if os.path.exists(part):
        os.remove(part)
    return False, note


def local_index():
    """Files already on disk under the input folders (a Kaggle dataset uploaded by hand), by file name."""
    idx = {}
    for root in INPUT_DIRS:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath[len(root):].count(os.sep) >= 5:
                dirnames[:] = []
            for fn in filenames:
                idx.setdefault(fn, os.path.join(dirpath, fn))
    return idx

In [ ]:
# ---- 4. Gaussian output parser --------------------------------------------------------------------------------
try:
    import numpy as np
except Exception:                                             # fingerprints are skipped without numpy
    np = None

PERIODIC = ('X H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Ge As Se Br Kr Rb Sr Y Zr '
            'Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os Ir Pt '
            'Au Hg Tl Pb Bi Po At Rn').split()
SYMBOL_Z = {s: i for i, s in enumerate(PERIODIC)}

# Every pattern starts with literal text, so the search runs at C speed on outputs of hundreds of megabytes.
G_ROUTE = re.compile(rb'\n -{5,}[ \t]*\n((?: #[^\n]*\n)(?: [^\n]*\n)*?) -{5,}[ \t]*\n')
G_VER = re.compile(rb'Gaussian (\d\d):\s+(\S+)')
G_SCF = re.compile(rb'SCF Done:\s+E\((\S+?)\)\s*=\s*(-?\d+\.\d+(?:[DEde][+-]?\d+)?)\s+A\.U\. after\s+(\d+) cycles')
G_ISO = re.compile(rb'\n[ \t]*(\d+)[ \t]+([A-Za-z]{1,3})[ \t]+Isotropic =[ \t]*(-?\d*\.\d+|\*+)[ \t]+Anisotropy =[ \t]*(-?\d*\.\d+|\*+)')
G_XYZ = re.compile(rb'[ \t]*(\d+)[ \t]+(-?\d+)[ \t]+(-?\d+)[ \t]+(-?\d+\.\d+)[ \t]+(-?\d+\.\d+)[ \t]+(-?\d+\.\d+)[ \t]*\n')
G_CHG = re.compile(rb'Charge\s*=\s*(-?\d+)\s+Multiplicity\s*=\s*(\d+)')
G_NAT = re.compile(rb'NAtoms=\s*(\d+)')
G_SOLV = re.compile(rb'Solvent\s*:\s*([^,\n]+),\s*Eps\s*=\s*(\d+\.\d+)')
G_CPU = re.compile(rb'Job cpu time:\s+(\d+) days\s+(\d+) hours\s+(\d+) minutes\s+([\d.]+) seconds')
G_NPROC = re.compile(rb'Will use up to\s+(\d+) processors')
G_GIBBS = re.compile(rb'Sum of electronic and thermal Free Energies=\s*(-?\d+\.\d+)')
G_ZPE = re.compile(rb'Sum of electronic and zero-point Energies=\s*(-?\d+\.\d+)')
G_FREQ = re.compile(rb'\n Frequencies --((?:[ \t]+-?\d+\.\d+)+)')


def _f(b):
    return float(b.replace(b'D', b'E').replace(b'd', b'e'))


def sniff_program(blob):
    head = blob[:60000]
    if b'Entering Gaussian System' in head or b'Gaussian, Inc.' in head or G_VER.search(head):
        return 'gaussian'
    if b'Northwest Computational Chemistry Package' in head:
        return 'nwchem'
    if b'O   R   C   A' in head:
        return 'orca'
    if b'BatchMin' in head or b'MacroModel' in head or b'Schrodinger' in head or b'Schr\xc3\xb6dinger' in head:
        return 'macromodel'
    if b'TINKER' in head or b'Tinker' in head:
        return 'tinker'
    if b'Jaguar' in head:
        return 'jaguar'
    return 'unknown'


def moments(xyz, z=None):
    """Principal moments of the point set (unit masses), all atoms and heavy atoms: a rotation-free conformer fingerprint."""
    if np is None or len(xyz) < 9:
        return None, None
    a = np.asarray(xyz, float).reshape(-1, 3)
    out = []
    for sel in (None, [i for i, zz in enumerate(z or []) if zz and zz > 1]):
        b = a if sel is None else (a[sel] if len(sel) >= 3 else None)
        if b is None:
            out.append(None)
            continue
        c = b - b.mean(axis=0)
        r2 = (c * c).sum()
        w = np.linalg.eigvalsh(r2 * np.eye(3) - c.T @ c)
        out.append([round(float(v), 3) for v in w])
    return out[0], out[1]


def _orientations(blob, lo, hi):
    """Positions and kinds of the geometry tables between lo and hi."""
    out = []
    i = blob.find(b' orientation:', lo, hi)
    while i >= 0:
        w = blob[max(lo, i - 8):i]
        kind = 'Standard' if w.endswith(b'Standard') else 'Input' if w.endswith(b'Input') else 'Z-Matrix' if w.endswith(b'Z-Matrix') else None
        if kind:
            out.append((i, kind))
        i = blob.find(b' orientation:', i + 13, hi)
    return out


def _orientation(blob, pos, hi):
    """Rows of the geometry table whose title is at pos: (atomic numbers, flat xyz)."""
    p = pos
    for _ in range(5):                                        # title line, rule, two header lines, rule
        q = blob.find(b'\n', p, hi)
        if q < 0:
            return [], []
        p = q + 1
    z, xyz = [], []
    while True:
        m = G_XYZ.match(blob, p, hi)
        if not m:
            break
        z.append(int(m.group(2)))
        xyz.extend((round(float(m.group(4)), 4), round(float(m.group(5)), 4), round(float(m.group(6)), 4)))
        p = m.end()
    return z, xyz


def _last(pattern, blob, lo, hi):
    m = None
    for m in pattern.finditer(blob, lo, hi):
        pass
    return m


def parse_gaussian(blob):
    """One record per job step (route section) of a Gaussian output. Returns (core records, extra records)."""
    if b'\r\n' in blob:
        blob = blob.replace(b'\r\n', b'\n')
    n = len(blob)
    ver = G_VER.search(blob, 0, min(n, 200000))
    nproc = G_NPROC.search(blob, 0, min(n, 200000))
    routes = list(G_ROUTE.finditer(blob))
    starts = [m.start() for m in routes] or [0]
    tail = blob[-400:].rstrip().split(b'\n')[-1].decode('latin-1', 'replace').strip()[:100]
    core, extra = [], []
    for k, lo in enumerate(starts):
        hi = starts[k + 1] if k + 1 < len(starts) else n
        rec = {'job': k, 'njobs': len(starts)}
        if ver:
            rec['ver'] = (ver.group(1) + b' ' + ver.group(2)).decode('latin-1')
        if nproc:
            rec['nproc'] = int(nproc.group(1))
        if routes:
            lines = routes[k].group(1).split(b'\n')
            rec['route'] = re.sub(r'\s+', ' ', ''.join(ln[1:].decode('latin-1', 'replace') for ln in lines if ln)).strip()[:600]
        m = G_CHG.search(blob, lo, hi)
        if m:
            rec['chg'], rec['mult'] = int(m.group(1)), int(m.group(2))
        m = G_NAT.search(blob, lo, hi)
        if m:
            rec['nat'] = int(m.group(1))
        i = blob.find(b' basis functions,', lo, hi)
        if i >= 0:
            word = blob[max(lo, i - 12):i].split()
            if word and word[-1].isdigit():
                rec['nbf'] = int(word[-1])
        if blob.find(b'Solvent', lo, hi) >= 0:
            m = G_SOLV.search(blob, lo, hi)
            if m:
                rec['solv'], rec['eps'] = m.group(1).decode('latin-1').strip(), float(m.group(2))
        scf, n_scf = None, 0
        for scf in G_SCF.finditer(blob, lo, hi):
            n_scf += 1
        if scf is not None:
            rec['scf'], rec['scf_m'], rec['n_scf'], rec['scf_cyc'] = _f(scf.group(2)), scf.group(1).decode('latin-1'), n_scf, int(scf.group(3))
        if blob.find(b'Sum of electronic', lo, hi) >= 0:
            m = _last(G_GIBBS, blob, lo, hi)
            if m is not None:
                rec['gibbs'] = float(m.group(1))
            m = _last(G_ZPE, blob, lo, hi)
            if m is not None:
                rec['e_zpe'] = float(m.group(1))
        if blob.find(b' Frequencies --', lo, hi) >= 0:
            freqs = [float(x) for m in G_FREQ.finditer(blob, lo, hi) for x in m.group(1).split()]
            if freqs:
                rec['n_freq'], rec['n_imag'] = len(freqs), sum(1 for x in freqs if x < 0)
        rec['norm'] = blob.count(b'Normal termination of Gaussian', lo, hi)
        if blob.find(b'Error termination', lo, hi) >= 0:
            rec['err'] = 1
        if blob.find(b'Optimization completed', lo, hi) >= 0:
            rec['opt_done'] = 1
        elif blob.find(b'Optimization stopped', lo, hi) >= 0 or blob.find(b'Number of steps exceeded', lo, hi) >= 0:
            rec['opt_done'] = 0
        cpu = sum(int(m.group(1)) * 86400 + int(m.group(2)) * 3600 + int(m.group(3)) * 60 + float(m.group(4)) for m in G_CPU.finditer(blob, lo, hi))
        if cpu:
            rec['cpu_s'] = round(cpu, 1)
        if k == len(starts) - 1:
            rec['tail'] = tail
        # NMR shieldings: the last tensor table of this step
        h = blob.rfind(b'Magnetic shielding tensor (ppm)', lo, hi)
        el = iso = aniso = None
        if h >= 0:
            el, iso, aniso = [], [], []
            for m in G_ISO.finditer(blob, h, hi):
                if int(m.group(1)) != len(el) + 1:
                    break
                el.append(m.group(2).decode('latin-1'))
                iso.append(None if m.group(3).startswith(b'*') else float(m.group(3)))
                aniso.append(None if m.group(4).startswith(b'*') else float(m.group(4)))
            rec['iso'] = iso
        # geometries: first table as given, last table (standard orientation if printed)
        oris = _orientations(blob, lo, hi)
        xrec = {'job': k}
        if oris:
            z1, xyz1 = _orientation(blob, oris[0][0], hi)
            std = [o for o in oris if o[1] == 'Standard'] or [o for o in oris if o[1] == oris[0][1]]
            n_geom = max(len([o for o in oris if o[1] == kind]) for kind in ('Input', 'Standard', 'Z-Matrix'))
            rec['n_geom'] = n_geom
            if z1:
                rec.setdefault('nat', len(z1))
                rec['pm'], rec['pmh'] = moments(xyz1, z1)
                xrec.update(z=z1, first_kind=oris[0][1], first=xyz1)
                if el is None:
                    el = [PERIODIC[v] if 0 <= v < len(PERIODIC) else 'X' for v in z1]
            if n_geom > 1:                                    # an optimisation: keep the final geometry as well
                z2, xyz2 = _orientation(blob, std[-1][0], hi)
                if z2:
                    rec['pm_last'], rec['pmh_last'] = moments(xyz2, z2)
                    xrec.update(last_kind=std[-1][1], last=xyz2)
        if el:
            rec['el'] = ' '.join(el)
        if aniso:
            xrec['aniso'] = aniso
        core.append({a: b for a, b in rec.items() if b is not None})
        if len(xrec) > 1:
            extra.append(xrec)
    return core, extra


def job_class(route):
    r = (route or '').lower()
    if 'nmr' in r:
        return 'nmr'
    if re.search(r'\bopt\b|opt=|opt\(', r):
        return 'opt+freq' if 'freq' in r else 'opt'
    if 'freq' in r:
        return 'freq'
    return 'sp' if r else 'no route'

In [ ]:
# ---- 5. Maestro (.mae) parser ---------------------------------------------------------------------------------
MAE_TOK = re.compile(r'"(?:[^"\\]|\\.)*"|#[^#\n]*#|[{}]|:::|[^\s{}]+')
MAE_CT = re.compile(r'^[ \t]*([fp])_m_ct[ \t]*\{', re.M)
MAE_IDX = re.compile(r'^(\w+)\[(\d+)\]$')


def _mae_val(name, tok):
    if tok.startswith('"') and tok.endswith('"') and len(tok) >= 2:
        tok = tok[1:-1].replace('\\"', '"').replace('\\\\', '\\')
    if tok in ('<>', ''):
        return None
    try:
        if name.startswith('r_'):
            return float(tok)
        if name.startswith(('i_', 'b_')):
            return int(tok)
    except ValueError:
        pass
    return tok[:200]


def _mae_block(toks, i):
    """Parse one block starting at its opening brace. Returns (properties, {sub-block name: (columns, rows)}, next index)."""
    assert toks[i] == '{'
    i += 1
    names = []
    while toks[i] != ':::':
        names.append(toks[i])
        i += 1
    i += 1
    props = {n: _mae_val(n, toks[i + k]) for k, n in enumerate(names)}
    i += len(names)
    subs = {}
    while toks[i] != '}':
        name = toks[i]
        i += 1
        m = MAE_IDX.match(name)
        if m:
            n_rows = int(m.group(2))
            assert toks[i] == '{'
            i += 1
            cols = []
            while toks[i] != ':::':
                cols.append(toks[i])
                i += 1
            i += 1
            width = len(cols) + 1
            rows = [toks[i + r * width:i + (r + 1) * width] for r in range(n_rows)]
            i += n_rows * width
            if toks[i] == ':::':
                i += 1
            while toks[i] != '}':                             # nested indexed sub-blocks are skipped
                if toks[i] == '{':
                    depth = 1
                    i += 1
                    while depth:
                        depth += {'{': 1, '}': -1}.get(toks[i], 0)
                        i += 1
                else:
                    i += 1
            i += 1
            subs[m.group(1)] = (cols, rows)
        else:
            _, _, i = _mae_block(toks, i)
    return props, subs, i + 1


def _mae_ct(kind, chunk, last_z):
    """One structure block (text starting at its opening brace) -> dict, and the atomic numbers to inherit."""
    toks = [t for t in MAE_TOK.findall(chunk) if not (t.startswith('#') and t.endswith('#') and len(t) > 1)]
    props, subs, _ = _mae_block(toks, 0)
    ct = {'full': kind == 'f', 'props': props}
    if 'm_atom' in subs:
        cols, rows = subs['m_atom']
        ix = {c: j + 1 for j, c in enumerate(cols)}
        if all(c in ix for c in ('r_m_x_coord', 'r_m_y_coord', 'r_m_z_coord')):
            xyz = []
            for r in rows:
                xyz.extend((round(float(r[ix['r_m_x_coord']]), 4), round(float(r[ix['r_m_y_coord']]), 4), round(float(r[ix['r_m_z_coord']]), 4)))
            ct['xyz'] = xyz
        if 'i_m_atomic_number' in ix:
            last_z = [int(r[ix['i_m_atomic_number']]) for r in rows]
        if 'i_m_formal_charge' in ix:
            ct['charge'] = sum(int(r[ix['i_m_formal_charge']]) for r in rows)
        if 'i_m_mmod_type' in ix and 'i_m_atomic_number' not in ix:
            ct['mmod_types'] = [int(r[ix['i_m_mmod_type']]) for r in rows]
        ct['n_atoms'] = len(rows)
        if last_z is not None and len(last_z) == len(rows):
            ct['z'] = last_z
    return ct, last_z


def iter_mae(fh, block=8 << 20):
    """Structures of a Maestro file, read from a binary stream one at a time (partial structures inherit the atom list)."""
    buf, last_z, pending = '', None, None
    while True:
        raw = fh.read(block)
        buf += raw.decode('latin-1')
        marks = list(MAE_CT.finditer(buf))
        if pending is None and marks:
            buf = buf[marks[0].start():]
            marks = list(MAE_CT.finditer(buf))
            pending = True
        limit = len(marks) if not raw else len(marks) - 1     # the last block may continue in the next read
        for k in range(max(limit, 0)):
            end = marks[k + 1].start() if k + 1 < len(marks) else len(buf)
            ct, last_z = _mae_ct(marks[k].group(1), buf[marks[k].end() - 1:end], last_z)
            yield ct
        if not raw:
            return
        if marks:
            buf = buf[marks[-1].start():]
        elif len(buf) > 4096:
            buf = buf[-4096:]                                 # header before the first structure: keep only a tail


def parse_mae(text):
    """Structures of a Maestro file given as text. Returns a list of dicts: props, n_atoms, z, xyz, charge, full."""
    return list(iter_mae(io.BytesIO(text.encode('latin-1'))))


def mae_energy_key(props):
    """The property that holds the conformer energy (kJ/mol in MacroModel files)."""
    keys = [k for k in props if k.startswith('r_mmod_Potential_Energy')] or \
           [k for k in props if k.startswith('r_') and 'energy' in k.lower() and 'relative' not in k.lower()]
    return keys[0] if keys else None

In [ ]:
# ---- 6. Reading archives and sorting their members --------------------------------------------------------------
QM_EXT = ('.out', '.log')
TEXT_EXT = ('.sdf', '.mol', '.mol2', '.txt', '.dp4', '.csv', '.tsv', '.json', '.md', '.xml', '.nmr', '.py', '.cfg', '.ini',
            '.yaml', '.yml', '.dat', '.xyz', '.smi', '.inchi', '.cdxml', '.html', '.readme')
BRUKER_DIRS = ('pdata', 'proton', 'carbon')
BRUKER_FILES = {'fid', 'ser', 'acqu', 'acqus', 'acqu2', 'acqu2s', 'proc', 'procs', 'proc2', 'proc2s', '1r', '1i', '2rr', 'title',
                'pulseprogram', 'audita.txt', 'auditp.txt', 'format.temp', 'scon2', 'uxnmr.par', 'uxnmr.info', 'outd', 'peaklist.xml',
                'specpar', 'precom.output', 'shimvalues', 'cpdprg2', 'vtc_pid_settings', 'prosol_history', 'thumb.png'}


def archive_kind(name):
    n = name.lower()
    if n.endswith('.zip'):
        return 'zip'
    if n.endswith(('.tar', '.tar.gz', '.tgz', '.tar.bz2', '.tbz2', '.tar.xz', '.txz')):
        return 'tar'
    if n.endswith(('.7z', '.rar')):
        return '7z'
    return None


def split_ext(path):
    """(lower-case extension without a trailing .gz, gz flag)."""
    low = path.rsplit('/', 1)[-1].lower()
    if low.endswith('.tar.gz'):
        return '.tar.gz', False
    gz = low.endswith('.gz')
    if gz:
        low = low[:-3]
    return (os.path.splitext(low)[1] if '.' in low else ''), gz


def classify(path, size):
    low = path.lower()
    base = low.rsplit('/', 1)[-1]
    comps = low.split('/')[:-1]
    ext, _ = split_ext(path)
    if archive_kind(base):
        return 'archive'
    if base.endswith(('.mae', '.maegz', '.mae.gz')):
        return 'mae'
    if ext in QM_EXT:
        return 'qm'
    bruker = base in BRUKER_FILES or any(c in BRUKER_DIRS or 'nmr_folder' in c or 'nmr_data' in c for c in comps)
    if size <= SMALL_FILE_KB * 1024 and not bruker:
        if ext in TEXT_EXT:
            return 'small'
        if ext == '' and size > 0:                            # NMR descriptions have no extension (XNMR, SX_NMR)
            return 'small'
    return 'other'


class ZipSource:
    """Members of a zip file. Each worker process opens its own handle."""
    def __init__(self, path):
        self.path = path

    def members(self):
        with zipfile.ZipFile(self.path) as zf:
            return [(i.filename, i.file_size) for i in zf.infolist() if not i.is_dir()]

    def __getstate__(self):
        return {'path': self.path}

    def open(self):
        self._zf = zipfile.ZipFile(self.path)
        return self

    def close(self):
        zf = self.__dict__.pop('_zf', None)
        if zf is not None:
            zf.close()

    def stream(self, name):
        return self._zf.open(name)


class DirSource:
    """Files of a folder tree (an extracted tar, or a dataset uploaded by hand)."""
    def __init__(self, root):
        self.root = root

    def members(self):
        out = []
        for dirpath, _, filenames in os.walk(self.root):
            for fn in filenames:
                p = os.path.join(dirpath, fn)
                try:
                    out.append((os.path.relpath(p, self.root).replace(os.sep, '/'), os.path.getsize(p)))
                except OSError:
                    pass
        return sorted(out)

    def open(self):
        return self

    def close(self):
        pass

    def stream(self, name):
        return open(os.path.join(self.root, *name.split('/')), 'rb')


def read_member(src, name, size):
    """Bytes of a member (gunzipped if needed). Very large members: first 2 MB and last 64 MB. Returns (bytes, partial flag)."""
    big = BIG_MEMBER_MB * 1024 * 1024
    with src.stream(name) as fh:
        if name.lower().endswith(('.gz', '.maegz')) and not name.lower().endswith('.tar.gz'):
            fh = gzip.GzipFile(fileobj=fh)
            size = None
        if size is not None and size <= big:
            return fh.read(), False
        head = fh.read(2 << 20)
        tail = collections.deque()
        kept = 0
        total = len(head)
        while True:
            chunk = fh.read(8 << 20)
            if not chunk:
                break
            total += len(chunk)
            tail.append(chunk)
            kept += len(chunk)
            while kept - len(tail[0]) >= (64 << 20):
                kept -= len(tail.popleft())
        if total <= big:
            return head + b''.join(tail), False
        return head + b'\n' + b''.join(tail)[-(64 << 20):], True


def head_tail(blob, n_head=70, n_tail=25, width=200):
    lines = blob[:400000].decode('latin-1', 'replace').split('\n')
    if len(blob) <= 400000 and len(lines) <= n_head + n_tail:
        return '\n'.join(ln[:width] for ln in lines)
    tail_lines = blob[-60000:].decode('latin-1', 'replace').split('\n')[-n_tail:]
    return '\n'.join(ln[:width] for ln in lines[:n_head]) + '\n   [...]\n' + '\n'.join(ln[:width] for ln in tail_lines)


def work_chunk(args):
    """Parse a list of members of one source. Runs in a worker process."""
    src, tasks, prefix = args
    res = dict(g=[], gx=[], mae=[], maex=[], small=[], samples=[], errs=[], counts=collections.Counter())
    try:
        src.open()
    except Exception as exc:
        res['errs'].append((prefix, 'cannot open source: %r' % (exc,)))
        return res
    for name, shown, size, kind, want_sample in tasks:
        path = prefix + shown
        try:
            if kind == 'qm':
                blob, partial = read_member(src, name, size)
                prog = sniff_program(blob)
                res['counts']['qm:' + prog] += 1
                if want_sample or (prog != 'gaussian' and res['counts']['qm:' + prog] == 1):
                    res['samples'].append((path, '%s output (%s)' % (prog, human(len(blob))), head_tail(blob)))
                if prog == 'gaussian':
                    core, extra = parse_gaussian(blob)
                    for r in core:
                        r['path'], r['bytes'] = path, len(blob)
                        if partial:
                            r['partial_read'] = 1
                        res['counts']['job:' + job_class(r.get('route'))] += 1
                        if 'iso' in r:
                            res['counts']['jobs with shieldings'] += 1
                        if r.get('norm'):
                            res['counts']['jobs ended normally'] += 1
                    for x in extra:
                        x['path'] = path
                    res['g'].extend(core)
                    res['gx'].extend(extra)
            elif kind == 'mae':
                gz = name.lower().endswith(('.gz', '.maegz'))
                if want_sample:
                    with src.stream(name) as fh:
                        head = (gzip.GzipFile(fileobj=fh) if gz else fh).read(60000)
                    res['samples'].append((path, 'Maestro file (%s)' % human(size), head_tail(head, 90, 0)))
                recs, geoms, ekey, emin = [], [], None, None
                with src.stream(name) as fh:
                    for j, c in enumerate(iter_mae(gzip.GzipFile(fileobj=fh) if gz else fh)):
                        p = c['props']
                        if j == 0:
                            ekey = mae_energy_key(p)
                        e = p.get(ekey) if ekey else None
                        e = e if isinstance(e, (int, float)) else None
                        keep_props = p if j == 0 else {a: b for a, b in p.items() if a.startswith(('r_mmod', 'i_mmod', 'b_mmod')) or a == 's_m_title'}
                        rec = dict(path=path, ct=j, full=int(c['full']), n_atoms=c.get('n_atoms'), props=keep_props)
                        if 'charge' in c:
                            rec['charge'] = c['charge']
                        if ekey:
                            rec['e_key'] = ekey
                        if 'xyz' in c:
                            rec['pm'], rec['pmh'] = moments(c['xyz'], c.get('z'))
                            if e is not None and (emin is None or e < emin):
                                emin = e
                                geoms = [g for g in geoms if g[0] is None or g[0] - emin <= MAE_WINDOW_KJ]
                            if e is None or e - emin <= MAE_WINDOW_KJ:
                                x = dict(path=path, ct=j, xyz=c['xyz'])
                                if 'z' in c:
                                    x['z'] = c['z']
                                elif 'mmod_types' in c:
                                    x['mmod_types'] = c['mmod_types']
                                geoms.append((e, j, x))
                                if len(geoms) > 4 * MAE_MAX_GEOMS:       # keep memory bounded on very long files
                                    geoms = sorted(geoms, key=lambda g: (g[0] is None, g[0] or 0.0, g[1]))[:MAE_MAX_GEOMS]
                        recs.append({a: b for a, b in rec.items() if b is not None})
                geoms = sorted(geoms, key=lambda g: (g[0] is None, g[0] or 0.0, g[1]))[:MAE_MAX_GEOMS]
                for r in recs:
                    r['n_ct'] = len(recs)
                res['mae'].extend(recs)
                res['maex'].extend(x for _, _, x in sorted(geoms, key=lambda g: g[1]))
                res['counts']['mae files'] += 1
                res['counts']['mae structures'] += len(recs)
            elif kind == 'small':
                blob, _ = read_member(src, name, size)
                res['small'].append((path, blob))
                if want_sample:
                    res['samples'].append((path, 'small file (%s)' % human(len(blob)), head_tail(blob, 60, 10)))
            elif want_sample:
                with src.stream(name) as fh:
                    blob = fh.read(60000)
                res['samples'].append((path, 'first bytes of %s' % human(size), head_tail(blob, 40, 0)))
        except Exception as exc:
            res['errs'].append((path, '%s: %s' % (type(exc).__name__, str(exc)[:200])))
            res['counts']['members that failed to parse'] += 1
    src.close()
    return res


def join_split_zip(folder, stem, parts):
    """Turn stem.z01 ... stem.zip into one ordinary zip. Returns the path or raises."""
    joined = os.path.join(folder, stem + '__joined.zip')
    last = os.path.join(folder, stem + '.zip')
    if shutil.which('zip'):
        p = subprocess.run(['zip', '-q', '-s', '0', last, '--out', joined], cwd=folder, capture_output=True, text=True)
        if p.returncode == 0 and os.path.exists(joined) and zipfile.is_zipfile(joined):
            return joined
        log('   zip -s 0 failed:', (p.stderr or p.stdout)[:200])
    # pure Python: concatenate the volumes and correct the member offsets by volume
    ordered = sorted([p for p in parts if not p.lower().endswith('.zip')]) + [last]
    sizes = [os.path.getsize(p) for p in ordered]
    with open(joined, 'wb') as out:
        for p in ordered:
            with open(p, 'rb') as fh:
                shutil.copyfileobj(fh, out, 1 << 22)
    zf = zipfile.ZipFile(joined)                              # raises for ZIP64 multi-volume sets
    starts = [sum(sizes[:k]) for k in range(len(sizes))]
    fixed = os.path.join(folder, stem + '__fixed.zip')
    with zipfile.ZipFile(fixed, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as dst:
        for info in zf.infolist():
            if info.is_dir():
                continue
            info.header_offset = info.header_offset - starts[-1] + starts[min(info.volume, len(starts) - 1)]
            if hasattr(info, '_end_offset'):
                info._end_offset = None                       # the guard was computed for the uncorrected offsets
            with zf.open(info) as src_fh, dst.open(info.filename, 'w', force_zip64=True) as dst_fh:
                shutil.copyfileobj(src_fh, dst_fh, 1 << 22)
    zf.close()
    os.remove(joined)
    return fixed


def extract_tar(path, dest):
    os.makedirs(dest, exist_ok=True)
    root = os.path.realpath(dest)
    with tarfile.open(path) as tf:
        for m in tf:
            target = os.path.realpath(os.path.join(dest, m.name))
            if not m.isfile() or not (target == root or target.startswith(root + os.sep)):
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with tf.extractfile(m) as src_fh, open(target, 'wb') as dst_fh:
                shutil.copyfileobj(src_fh, dst_fh, 1 << 22)
    return dest

In [ ]:
# ---- 7. Result writers ----------------------------------------------------------------------------------------
class PartWriter:
    """Gzipped JSON lines, split into parts below the size of one result zip."""
    def __init__(self, folder, stem):
        self.folder, self.stem, self.k, self.n, self.fh, self.raw = folder, stem, 0, 0, None, None
        self.limit = int(PART_MB * 0.85 * 1024 * 1024)

    def _open(self):
        self.k += 1
        self.raw = open(os.path.join(self.folder, '%s_%03d.jsonl.gz' % (self.stem, self.k)), 'wb')
        self.fh = gzip.GzipFile(fileobj=self.raw, mode='wb', compresslevel=6, mtime=0)

    def write(self, obj):
        if self.fh is None:
            self._open()
        self.fh.write((json.dumps(obj, separators=(',', ':')) + '\n').encode('utf-8'))
        self.n += 1
        if self.n % 200 == 0 and self.raw.tell() > self.limit:
            self.close()

    def close(self):
        if self.fh is not None:
            self.fh.close()
            self.raw.close()
            self.fh = self.raw = None


class DepositOutput:
    def __init__(self, key):
        self.key = key
        self.g, self.gx = PartWriter(CORE, 'gaussian_' + key), PartWriter(EXTRA, 'geom_' + key)
        self.mae, self.maex = PartWriter(CORE, 'mae_' + key), PartWriter(EXTRA, 'maegeom_' + key)
        self.small_k, self.small_zip = 0, None
        self.small_seen, self.small_bytes, self.small_index = {}, 0, []
        self.paths = gzip.open(os.path.join(EXTRA, 'paths_%s.tsv.gz' % key), 'wt', encoding='utf-8')
        self.dirs = collections.OrderedDict()
        self.samples, self.sample_kinds = [], set()
        self.counts = collections.Counter()
        self.errors = []
        self.census = collections.Counter()

    def note_member(self, path, size):
        self.paths.write('%s\t%d\n' % (path, size))
        d, _, base = path.rpartition('/')
        e = self.dirs.setdefault(d, dict(n=0, bytes=0, ext=collections.Counter(), ex=[]))
        e['n'] += 1
        e['bytes'] += size
        e['ext'][split_ext(path)[0] or '(none)'] += 1
        if len(e['ex']) < 3:
            e['ex'].append(base)

    def absorb(self, res):
        for r in res['g']:
            self.g.write(r)
            parts = r['path'].split('!/')[-1].split('/')
            self.census[('/'.join(parts[:-1])[-70:], job_class(r.get('route')), r.get('scf_m', '-'))] += 1
        for r in res['gx']:
            self.gx.write(r)
        for r in res['mae']:
            self.mae.write(r)
        for r in res['maex']:
            self.maex.write(r)
        for path, blob in res['small']:
            digest = hashlib.md5(blob).hexdigest()
            first = self.small_seen.get(digest)
            stored = 0
            if first is None and self.small_bytes + len(blob) <= SMALL_TOTAL_MB * 1024 * 1024:
                self.small_seen[digest] = path
                self.small_bytes += len(blob)
                if self.small_zip is None or self.small_zip.fp.tell() > PART_MB * 0.8 * 1024 * 1024:
                    if self.small_zip is not None:
                        self.small_zip.close()
                    self.small_k += 1
                    self.small_zip = zipfile.ZipFile(os.path.join(CORE, 'small_%s_%03d.zip' % (self.key, self.small_k)), 'w', zipfile.ZIP_DEFLATED)
                self.small_zip.writestr(path.replace('!/', '/'), blob)
                stored = 1
            self.small_index.append((path, len(blob), digest, stored, first or ''))
        self.samples.extend(res['samples'])
        self.counts.update(res['counts'])
        self.errors.extend(res['errs'])

    def close(self):
        for w in (self.g, self.gx, self.mae, self.maex):
            w.close()
        if self.small_zip is not None:
            self.small_zip.close()
        self.paths.close()
        with gzip.open(os.path.join(CORE, 'small_index_%s.tsv.gz' % self.key), 'wt', encoding='utf-8') as fh:
            fh.write('path\tbytes\tmd5\tstored\tsame_content_as\n')
            for row in self.small_index:
                fh.write('%s\t%d\t%s\t%d\t%s\n' % row)
        with gzip.open(os.path.join(CORE, 'dirs_%s.tsv.gz' % self.key), 'wt', encoding='utf-8') as fh:
            fh.write('folder\tfiles\tbytes\textensions\texamples\n')
            for d, e in self.dirs.items():
                fh.write('%s\t%d\t%d\t%s\t%s\n' % (d, e['n'], e['bytes'], json.dumps(dict(e['ext'].most_common())), ' | '.join(e['ex'])))
        with gzip.open(os.path.join(CORE, 'samples_%s.txt.gz' % self.key), 'wt', encoding='utf-8') as fh:
            for path, what, text in self.samples:
                fh.write('=' * 110 + '\n%s\n%s\n' % (path, what) + '-' * 110 + '\n' + text + '\n\n')
        with open(os.path.join(CORE, 'errors_%s.txt' % self.key), 'w', encoding='utf-8') as fh:
            for path, msg in self.errors:
                fh.write('%s\t%s\n' % (path, msg))

In [ ]:
# ---- 8. Processing one source (archive or folder) -----------------------------------------------------------------
def sample_kind(path, kind):
    comps = path.split('/')
    parent = comps[-2].lower() if len(comps) > 1 else ''
    return (kind, split_ext(path)[0], parent if parent in ('nmr', 'opt', 'e', 'mm', 'opt_e') else '')


def process_source(src, prefix, out, depth=0):
    """Inventory every member, parse what is recognised, recurse into nested archives."""
    try:
        members = src.members()
    except Exception as exc:
        out.errors.append((prefix, 'cannot list: %r' % (exc,)))
        log('   cannot read', prefix, '|', repr(exc)[:160])
        return
    tasks, nested = [], []
    for name, size in members:
        shown = name.replace('\\', '/')                       # zips written on Windows may use backslashes
        out.note_member(prefix + shown, size)
        kind = classify(shown, size)
        out.counts['members'] += 1
        out.counts['kind:' + kind] += 1
        sk = sample_kind(shown, kind)
        want = sk not in out.sample_kinds and len(out.sample_kinds) < 80
        if want:
            out.sample_kinds.add(sk)
        if kind == 'archive':
            nested.append((name, shown, size))
        elif kind in ('qm', 'mae', 'small') or want:
            tasks.append((name, shown, size, kind, want))
    # chunks of roughly 48 MB or 300 members
    chunks, cur, cur_bytes = [], [], 0
    for t in tasks:
        cur.append(t)
        cur_bytes += t[2]
        if cur_bytes >= (48 << 20) or len(cur) >= 300:
            chunks.append(cur)
            cur, cur_bytes = [], 0
    if cur:
        chunks.append(cur)
    jobs = [(src, c, prefix) for c in chunks]
    done = False
    if N_WORKERS > 1 and len(jobs) > 1:
        try:
            ctx = multiprocessing.get_context('fork')
            with ctx.Pool(N_WORKERS) as pool:
                for res in pool.imap(work_chunk, jobs):
                    out.absorb(res)
            done = True
        except Exception as exc:
            log('   parallel run failed (%s); repeating this source on one core' % repr(exc)[:120])
            out.errors.append((prefix, 'parallel run failed: %r' % (exc,)))
    if not done:
        for job in jobs:
            out.absorb(work_chunk(job))
    for name, shown, size in nested:
        if depth >= 3:
            out.errors.append((prefix + shown, 'nested archive deeper than 3 levels: not opened'))
            continue
        if free_gb(TMP) * 1024 ** 3 < size * 6 + (1 << 30):
            out.errors.append((prefix + shown, 'not enough scratch space to open nested archive'))
            continue
        tmp = os.path.join(TMP, 'nested_%d_%s' % (depth, safe_name(shown)))
        try:
            with src.open().stream(name) as fh, open(tmp, 'wb') as dst:
                shutil.copyfileobj(fh, dst, 1 << 22)
            src.close()
            process_file(tmp, prefix + shown + '!/', out, depth + 1, display=shown.rsplit('/', 1)[-1])
        except Exception as exc:
            out.errors.append((prefix + shown, 'nested archive: %s' % str(exc)[:200]))
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)


def process_file(path, prefix, out, depth=0, display=None):
    """Open one file from the repository (or a nested one) according to its type."""
    name = display or os.path.basename(path)
    kind = archive_kind(name)
    if kind == 'zip':
        if not zipfile.is_zipfile(path):
            raise RuntimeError('not a readable zip file')
        process_source(ZipSource(path), prefix, out, depth)
    elif kind == 'tar':
        dest = os.path.join(TMP, 'untar_%d_%s' % (depth, safe_name(name)))
        try:
            process_source(DirSource(extract_tar(path, dest)), prefix, out, depth)
        finally:
            shutil.rmtree(dest, ignore_errors=True)
    elif kind == '7z':
        exe = shutil.which('7z') or shutil.which('7za') or shutil.which('7zr')
        if not exe:
            raise RuntimeError('7z archive and no 7z program on this machine')
        dest = os.path.join(TMP, 'un7z_%d_%s' % (depth, safe_name(name)))
        try:
            p = subprocess.run([exe, 'x', '-y', '-o' + dest, path], capture_output=True, text=True)
            if p.returncode != 0:
                raise RuntimeError('7z failed: ' + (p.stderr or p.stdout)[-200:])
            process_source(DirSource(dest), prefix, out, depth)
        finally:
            shutil.rmtree(dest, ignore_errors=True)
    else:                                                     # a loose file (for example References.txt)
        folder = os.path.join(TMP, 'loose_%d' % depth)
        shutil.rmtree(folder, ignore_errors=True)
        os.makedirs(folder)
        try:
            if os.path.getsize(path) <= (1 << 30):
                shutil.copy(path, os.path.join(folder, name))
                process_source(DirSource(folder), prefix, out, depth)
            else:
                out.note_member(prefix + name, os.path.getsize(path))
                out.counts['members'] += 1
                out.counts['kind:other'] += 1
        finally:
            shutil.rmtree(folder, ignore_errors=True)

In [ ]:
# ---- 9. Run: every deposit -------------------------------------------------------------------------------------
SUMMARY = dict(version=NOTEBOOK_VERSION, started=time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), deposits=[])


def online():
    for url in ('https://pypi.org/simple/', 'https://www.kaggle.com/'):
        try:
            SESSION.get(url, timeout=10)
            return True
        except requests.RequestException:
            pass
    return False


if not LOCAL_SOURCES and not os.environ.get('CS_API_BASES') and not online():
    log('!' * 100)
    log('NO INTERNET CONNECTION. In the right panel open Settings and switch Internet on, then run again.')
    log('!' * 100)
SUMMARY['online'] = None if os.environ.get('CS_API_BASES') else online()
LOCAL = local_index()
if LOCAL:
    log('files found under the input folders:', len(LOCAL))


def run_deposit(dep):
    key = dep['key']
    info = dict(key=key, doi=dep.get('doi'), title=dep.get('title'), mode=dep.get('mode'), files=[])
    SUMMARY['deposits'].append(info)
    log('=' * 100)
    log('DEPOSIT', key, '|', dep.get('title'), '| doi', dep.get('doi'), '| mode', dep.get('mode'))
    out = DepositOutput(key)
    try:
        manual = LOCAL_SOURCES.get(key) or []
        if manual:
            info['listing'] = 'manual sources'
            for p in manual:
                frec = dict(name=os.path.basename(p.rstrip('/')), source=p)
                info['files'].append(frec)
                try:
                    if os.path.isdir(p):
                        process_source(DirSource(p), os.path.basename(p.rstrip('/')) + '/', out)
                    else:
                        process_file(p, os.path.basename(p) + '!/', out)
                    frec['status'] = 'parsed'
                except Exception as exc:
                    frec['status'] = 'failed: %s' % str(exc)[:200]
                    log('   FAILED', p, '|', str(exc)[:200])
            return
        files, how = list_record(dep)
        info['listing'] = how
        with open(os.path.join(CORE, 'files_%s.json' % key), 'w', encoding='utf-8') as fh:
            json.dump(dict(deposit=dep, listing=how, files=files), fh, indent=1)
        total = sum(f.get('size') or 0 for f in files)
        log('   listing: %s | %d files | %s in total' % (how, len(files), human(total)))
        for f in files:
            log('      %-52s %10s' % (f['name'][:52], human(f.get('size'))))
        info['n_files'], info['total_bytes'] = len(files), total
        known = {n: s for n, _, s in KNOWN_FILES.get(key, [])}
        for f in files:
            if f['name'] in known and f.get('size') not in (None, known[f['name']]):
                log('   note: %s is %s bytes in the API and %s on the record page' % (f['name'], f.get('size'), known[f['name']]))
        if not files:
            log('   NO FILE LIST. Manual route: download the files of doi', dep.get('doi'), 'in a browser, upload them as a Kaggle')
            log('   dataset, attach it to this notebook, set LOCAL_SOURCES = {"%s": ["/kaggle/input/<dataset>"]} and run again.' % key)
            return
        if dep.get('mode') != 'parse':
            for f in files:
                info['files'].append(dict(name=f['name'], size=f.get('size'), status='listed only'))
            return
        if total > MAX_DEPOSIT_GB * 1024 ** 3:
            log('   deposit is larger than MAX_DEPOSIT_GB = %s: listed only' % MAX_DEPOSIT_GB)
            for f in files:
                info['files'].append(dict(name=f['name'], size=f.get('size'), status='listed only (deposit too large)'))
            return
        # split zip sets: stem.z01, stem.z02, ..., stem.zip
        groups = collections.OrderedDict()
        for f in files:
            m = re.match(r'^(.*)\.z(\d{2,})$', f['name'], re.I)
            stem = m.group(1) if m else None
            groups.setdefault(('split', stem.lower()) if stem else ('single', f['name']), []).append(f)
        for (kind, gname), group in list(groups.items()):
            if kind == 'split':
                last = [g for g in files if g['name'].lower() == gname + '.zip']
                if last and ('single', last[0]['name']) in groups:
                    group.extend(groups.pop(('single', last[0]['name'])))
        for (kind, gname), group in groups.items():
            names = [f['name'] for f in group]
            frec = dict(name=' + '.join(names) if kind == 'split' else names[0], size=sum(f.get('size') or 0 for f in group))
            info['files'].append(frec)
            if ONLY_FILES and not any(n in ONLY_FILES for n in names):
                frec['status'] = 'skipped (not in ONLY_FILES)'
                continue
            if any(n in (dep.get('skip_names') or []) for n in names):
                frec['status'] = 'skipped (skip_names)'
                continue
            if hours() > DEADLINE_HOURS:
                frec['status'] = 'not started: time limit reached'
                continue
            if frec['size'] > MAX_FILE_GB * 1024 ** 3:
                frec['status'] = 'skipped: larger than MAX_FILE_GB'
                continue
            need = frec['size'] * (2.2 if kind == 'split' else 1.1) + (2 << 30)
            if free_gb(DL) * 1024 ** 3 < need:
                frec['status'] = 'skipped: not enough scratch space (%.1f GB free)' % free_gb(DL)
                log('   SKIP', frec['name'], '|', frec['status'])
                continue
            t1 = time.time()
            folder = os.path.join(DL, key, safe_name(gname))
            shutil.rmtree(folder, ignore_errors=True)
            os.makedirs(folder)
            try:
                paths, notes = [], []
                for f in group:
                    dest = os.path.join(folder, f['name'])
                    if f['name'] in LOCAL and (not f.get('size') or os.path.getsize(LOCAL[f['name']]) == f['size']):
                        dest, note = LOCAL[f['name']], 'taken from the input folder'
                    else:
                        ok, note = download(f, dest)
                        if not ok:
                            raise RuntimeError('download failed for %s: %s' % (f['name'], note))
                    paths.append(dest)
                    notes.append(note)
                frec['download'] = '; '.join(sorted(set(notes)))
                t2 = time.time()
                before = collections.Counter(out.counts)
                if kind == 'split':
                    for p in paths:                           # volumes taken from the input folder are copied next to the others
                        if os.path.dirname(p) != folder:
                            shutil.copy(p, os.path.join(folder, os.path.basename(p)))
                    last = [f['name'] for f in group if f['name'].lower().endswith('.zip')]
                    if not last:
                        raise RuntimeError('split zip set without its final .zip volume')
                    stem = last[0][:-4]
                    target = join_split_zip(folder, stem, [os.path.join(folder, n) for n in names])
                    process_file(target, stem + '.zip!/', out, display=stem + '.zip')
                else:
                    process_file(paths[0], names[0] + '!/' if archive_kind(names[0]) else '', out, display=names[0])
                delta = out.counts - before
                frec['status'] = 'parsed'
                frec['members'] = delta.get('members', 0)
                frec['gaussian_jobs'] = sum(v for k2, v in delta.items() if k2.startswith('job:'))
                frec['mae_structures'] = delta.get('mae structures', 0)
                log('   ok  %-40s %9s | download %4.0fs (%s) | parse %4.0fs | members %6d | Gaussian jobs %6d | Maestro structures %6d'
                    % (frec['name'][:40], human(frec['size']), t2 - t1, frec['download'], time.time() - t2,
                       frec['members'], frec['gaussian_jobs'], frec['mae_structures']))
            except Exception as exc:
                frec['status'] = 'failed: %s' % str(exc)[:300]
                out.errors.append((frec['name'], frec['status']))
                log('   FAILED', frec['name'], '|', str(exc)[:300])
                log('   ' + traceback.format_exc().strip().splitlines()[-1][:300])
            finally:
                shutil.rmtree(folder, ignore_errors=True)
    finally:
        out.close()
        info['counts'] = dict(sorted(out.counts.items()))
        info['n_errors'] = len(out.errors)
        info['census'] = [[d, c, m, n] for (d, c, m), n in sorted(out.census.items())][:4000]
        if out.counts:
            log('   totals:', ' | '.join('%s %d' % kv for kv in sorted(out.counts.items())))


for dep in DEPOSITS:
    if dep.get('mode') == 'off':
        continue
    try:
        run_deposit(dep)
    except Exception as exc:
        log('DEPOSIT', dep.get('key'), 'stopped by an error:', repr(exc)[:300])
        log(traceback.format_exc()[-1500:])
SUMMARY['hours'] = round(hours(), 2)
with open(os.path.join(CORE, 'summary.json'), 'w', encoding='utf-8') as fh:
    json.dump(SUMMARY, fh, indent=1)
log('all deposits done after %.2f h' % hours())

In [ ]:
# ---- 10. Package the results into zips of at most PART_MB --------------------------------------------------------
def pack(folder, stem):
    """Greedy packing of the files of a folder into stem_NN.zip in WORK. Large files get their own zip."""
    limit = int(PART_MB * 1024 * 1024)
    files = sorted((os.path.getsize(os.path.join(folder, f)), f) for f in os.listdir(folder))
    files = [(s, f) for s, f in files if s > 0 or f.endswith('.txt')]
    bins = []
    for size, f in sorted(files, key=lambda t: (-t[0], t[1])):
        for b in bins:
            if b['bytes'] + size <= limit:
                b['files'].append(f)
                b['bytes'] += size
                break
        else:
            bins.append(dict(files=[f], bytes=size))
    made = []
    for k, b in enumerate(bins, 1):
        target = os.path.join(WORK, '%s_%02d.zip' % (stem, k))
        with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for f in sorted(b['files']):
                stored = f.endswith(('.gz', '.zip'))
                zf.write(os.path.join(folder, f), f, zipfile.ZIP_STORED if stored else zipfile.ZIP_DEFLATED)
        made.append((os.path.basename(target), os.path.getsize(target), sorted(b['files'])))
    return made


for old in os.listdir(WORK):
    if re.match(r'^cs_(core|extra)_\d+\.zip$', old):
        os.remove(os.path.join(WORK, old))
log('packaging')
with open(os.path.join(CORE, 'MANIFEST.txt'), 'w', encoding='utf-8') as fh:
    fh.write('ConformalStereo Cambridge deposits, notebook %s, written %s\n\n' % (NOTEBOOK_VERSION, time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())))
    for label, folder in (('core', CORE), ('extra', EXTRA)):
        fh.write('[%s]\n' % label)
        for f in sorted(os.listdir(folder)):
            if f != 'MANIFEST.txt':
                fh.write('  %-46s %12d bytes\n' % (f, os.path.getsize(os.path.join(folder, f))))
core_zips = pack(CORE, 'cs_core')
extra_zips = pack(EXTRA, 'cs_extra')

print()
print('=' * 100)
print('RESULT FILES in', WORK)
for name, size, _ in core_zips:
    print('   SEND THIS   %-22s %10s' % (name, human(size)))
for name, size, _ in extra_zips:
    print('   keep        %-22s %10s   (conformer geometries; send only if asked)' % (name, human(size)))
print()
for info in SUMMARY['deposits']:
    c = info.get('counts') or {}
    st = collections.Counter((f.get('status') or '?').split(':')[0] for f in info.get('files', []))
    print('%-9s %-28s files: %s' % (info['key'], '(' + str(info.get('listing')) + ')', ', '.join('%d %s' % (n, s) for s, n in st.items()) or 'none'))
    if c:
        jobs = {k[4:]: v for k, v in c.items() if k.startswith('job:')}
        print('          Gaussian job steps: %d %s | with shieldings: %d | ended normally: %d' % (
            sum(jobs.values()), jobs, c.get('jobs with shieldings', 0), c.get('jobs ended normally', 0)))
        print('          Maestro files: %d with %d structures | small files copied: %d | members in all: %d | parse errors: %d' % (
            c.get('mae files', 0), c.get('mae structures', 0), c.get('kind:small', 0), c.get('members', 0), info.get('n_errors', 0)))
        other = {k[3:]: v for k, v in c.items() if k.startswith('qm:') and k != 'qm:gaussian'}
        if other:
            print('          other .out/.log files (not parsed):', other)
    for f in info.get('files', []):
        if str(f.get('status', '')).startswith(('failed', 'skipped', 'not started')):
            print('          !! %-40s %s' % (f['name'][:40], f['status'][:120]))
warn = []
for info in SUMMARY['deposits']:
    c = info.get('counts') or {}
    if info.get('mode') == 'parse' and not any(k.startswith('job:') for k in c):
        warn.append('%s: no Gaussian job was parsed' % info['key'])
    if str(info.get('listing', '')).startswith(('fallback', 'none')):
        warn.append('%s: the repository API did not answer; file list is %s' % (info['key'], info.get('listing')))
print()
print('WARNINGS:' if warn else 'No warnings.')
for w in warn:
    print('   -', w)
print('Finished in %.2f h.' % hours())